# NDR — Notebook 9-v3: RobustKD + giữ-retention (ULRL cosine-reg)

## Kết quả NB9-v2 (thật): firing 0.97→0.18, val 1.040 -> LB 259. Nút thắt = RETENTION 0.54.
## Research (dep-research) cho hướng B:
- **ULRL (2024):** cosine-similarity reg giữ hành vi benign khi unlearn -> thêm cosine-reg neo head gần model gốc.
- **RobustKD feature-variance:** cơ chế giữ retention chính; NB9-v2 dùng lam_fvar thấp -> sweep lam_fvar CAO.
- **Nhiều clean signal hơn:** tăng n_test (distill trên nhiều ảnh sạch) -> retention lên.

## Hướng B: kéo retention 0.54 -> mục tiêu 0.7+ mà giữ firing thấp
Sweep: sup_ratio THẤP (bớt suppress, đỡ hại streak) + lam_fvar CAO + n_test CAO + cosine-reg giữ head.

**Quy tắc nộp:** retention >= 0.65 VÀ val_macadd < 1.040 (neo NB9-v2).

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 8.6 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
# --- Confidence-fidelity diagnostic (proxy gan voi maCADD/clean model) ---
# maCADD phat theo |clean_score - depois_score| (A=10 bat doi xung). Firing/retention
# chi do CO/KHONG detect; cell nay do do KHOP CONFIDENCE voi clean model tren streak that,
# va residual confidence tren poison. Gan hon voi hanh vi top-1 (recovery + calibration).
def conf_fidelity(predictor, val, clean_ref, tta=False, floor=SUBMIT_THRESH):
    gaps=[]; pred_on_streak=[]; ref_on_streak=[]; missed=0; total_streak=0
    pois_res=[]; total_pois=0
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor,item['img']) if tta else predict_image(predictor,item['img']))
        keep = s>floor; b=(b[keep] if len(b) else b); s=(s[keep] if len(b) else s)
        rb,rs = clean_ref[str(i)]
        for k in range(len(rb)):
            total_streak+=1
            if len(b):
                ious=iou_matrix(rb[k:k+1],b)[0]; j=int(ious.argmax())
                if ious[j]>=MATCH_IOU:
                    gaps.append(abs(float(rs[k])-float(s[j]))); ref_on_streak.append(float(rs[k])); pred_on_streak.append(float(s[j])); continue
            gaps.append(float(rs[k])); missed+=1   # streak bi bo lo -> gap = toan bo conf ref (dong gop FN)
        pb=np.array(item['poison_boxes'],np.float32).reshape(-1,4)
        for k in range(len(pb)):
            total_pois+=1
            if len(b):
                ious=iou_matrix(pb[k:k+1],b)[0]
                if ious.max()>=MATCH_IOU: pois_res.append(float(s[int(ious.argmax())]))
    return dict(
        streak_conf_gap=float(np.mean(gaps)) if gaps else 0.0,
        streak_miss_rate=missed/max(total_streak,1),
        pred_conf_on_streak=float(np.mean(pred_on_streak)) if pred_on_streak else 0.0,
        ref_conf_on_streak=float(np.mean(ref_on_streak)) if ref_on_streak else 0.0,
        poison_residual_conf=float(np.mean(pois_res)) if pois_res else 0.0,
        poison_fire_frac=len(pois_res)/max(total_pois,1))

base_fid = conf_fidelity(poisoned_predictor, VAL, CLEAN_REF)
print('POISONED conf-fidelity:', {k:round(v,4) for k,v in base_fid.items()})
print('-> Muc tieu: streak_conf_gap THAP (khop conf clean model tren streak that),')
print('   poison_residual_conf ~0 (khong con FP poison). Ca hai giam = gan hanh vi top-1.')


POISONED conf-fidelity: {'streak_conf_gap': 0.0194, 'streak_miss_rate': 0.0, 'pred_conf_on_streak': 0.6822, 'ref_conf_on_streak': 0.6627, 'poison_residual_conf': 0.5937, 'poison_fire_frac': 0.9693}
-> Muc tieu: streak_conf_gap THAP (khop conf clean model tren streak that),
   poison_residual_conf ~0 (khong con FP poison). Ca hai giam = gan hanh vi top-1.


In [9]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. Baseline + neo LB

In [10]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
LB_ANCHOR_MACADD = 1.076   # local val_macadd của p05 (=LB 261.2)
print('POISONED baseline:', {k: round(v,4) for k,v in base_stats.items()})
print('Local neo (neo NB4 = LB258.7): val_macadd =', LB_ANCHOR_MACADD)
print('-> Chỉ nộp NB4 nếu val_macadd < %.3f VÀ retention >= 0.6' % LB_ANCHOR_MACADD)

POISONED baseline: {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
Local neo (neo NB4 = LB258.7): val_macadd = 1.076
-> Chỉ nộp NB4 nếu val_macadd < 1.076 VÀ retention >= 0.6


## 8. Retain-thật từ ảnh test (dùng cho distillation)

In [11]:
IMG_STORE = {}
def build_real_retain(n_test=250, retain_conf=0.4, n_poison_imgs=20, add_synth=True, seed=SEED):
    '''Retain set IN-DISTRIBUTION: nền ảnh test thật + pseudo-GT streak từ model gốc,
    TĂNG mật độ bằng synthetic streak có nhãn chuẩn (chống retain thưa gây under-train).
    Poison patch dán (không nhãn) = forget signal.'''
    rng = np.random.default_rng(seed); records = []; npos = 0; nposimg = 0
    pick = list(range(len(test_files))); rng.shuffle(pick); pick = pick[:n_test]
    for k in pick:
        p = test_files[k]; img = load_image(p); anns = []
        b, sc = predict_image(poisoned_predictor, img)
        rb = b[sc >= retain_conf] if len(b) else b
        for (x1,y1,x2,y2) in rb:
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
            x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[x1,y1,x2,y2],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        if add_synth:  # bơm thêm synthetic streak (nhãn CHẮC CHẮN đúng) lên nền test thật
            g = img[:,:,0].copy()
            for _ in range(rng.integers(1,4)):
                g, box = synth_streak_on(g, rng)
                x1,y1,x2,y2 = box
                if x2-x1>1 and y2-y1>1:
                    anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],
                                 'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
            img = np.repeat(g[:,:,None],3,axis=2)
        iid = 20_000_000 + int(k); IMG_STORE[iid] = img.astype(np.float32)
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    # ảnh poison-forget từ unlearn set
    for j in range(n_poison_imgs):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); anns=[]
        b, sc = predict_image(poisoned_predictor, img)
        pb = poison_boxes_by_stem.get(stem, np.zeros((0,4),np.float32))
        keep = sc >= retain_conf
        if len(b) and len(pb) and keep.any():
            keep = keep & (iou_matrix(b, pb).max(1) < POISON_OVERLAP_IOU)
        for (x1,y1,x2,y2) in (b[keep] if len(b) else b):
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        for _ in range(rng.integers(1,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem, ph, pw, rng)
            if spot: paste_patch(img, patch, spot[0], spot[1])
        iid = 21_000_000 + j; IMG_STORE[iid] = img.astype(np.float32); nposimg += 1
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    rng.shuffle(records)
    return records, npos, nposimg
IMG_STORE.clear()
_r,_np,_npi = build_real_retain(n_test=80, n_poison_imgs=10, seed=1)
print(f'probe retain-thật+synth: {len(_r)} ảnh | {_np} pseudo-GT streak ({_np/max(len(_r),1):.2f}/ảnh) | {_npi} ảnh poison-forget')
IMG_STORE.clear()

probe retain-thật+synth: 90 ảnh | 208 pseudo-GT streak (2.31/ảnh) | 10 ảnh poison-forget


## 9. Forward khả vi lấy cls logits (cho distillation)
Trả về list logits mỗi FPN level (chưa sigmoid). Teacher & student dùng chung hàm này.

### 8b. Detoxify: prune channel poison (cls+bbox) — RobustKD step 1

In [12]:
def _prune_subnet(model, seq, prune_frac, poison_xywh, tag):
    '''Prune channel phản hồi poison mạnh trong 1 subnet (cls hoặc bbox).'''
    convs=[m for m in seq if isinstance(m, nn.Conv2d)]
    stored={i:[] for i in range(len(convs))}; hooks=[]
    for i,ly in enumerate(convs):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts=[{'file_name':str(Path(UNLEARN_DIR)/f'{s}.png'),'image_id':int(s),'height':IMG_H,'width':IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im=load_image(d['file_name']); model([{'image':torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    total=0
    for li,layer in enumerate(convs):
        acts=stored[li]; C=acts[0].shape[1]; fg=torch.zeros(C); bg=torch.zeros(C); nf=nb_=0
        rng=np.random.default_rng(SEED)
        for act,d in zip(acts,dicts):
            a=act[0]; _,aH,aW=a.shape; sy,sx=aH/d['height'],aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'],[]):
                x1,y1=max(0,int(x*sx)),max(0,int(y*sy)); x2,y2=min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg+=a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw=max(1,int(16*sy)),max(1,int(16*sx)); ry,rx=rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg+=a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score=(fg-bg).numpy(); n_prune=max(1,int(len(score)*prune_frac)); bad=np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        total+=n_prune
    return total

def collect_and_prune(model, prune_frac, prune_box=True):
    model.eval(); total=0
    total += _prune_subnet(model, model.head.cls_subnet, prune_frac, poison_xywh, 'cls')
    if prune_box and hasattr(model.head,'bbox_subnet'):
        total += _prune_subnet(model, model.head.bbox_subnet, prune_frac, poison_xywh, 'box')
    return model, total

In [13]:
def cls_logits(model, x255, feats_out=None):
    '''x255: tensor [3,H,W] RGB 0-255. Trả list logits mỗi level; nếu feats_out là list, append FPN feats.'''
    nrm = ((x255 - model.pixel_mean) / model.pixel_std)[None]
    feats = model.backbone(nrm)
    fl = [feats[f] for f in model.head_in_features]
    if feats_out is not None: feats_out.extend([f for f in fl])
    logits, _ = model.head(fl)
    return logits

def load_teacher_detox(prune_frac=0.05, prune_box=True):
    '''RobustKD step1: teacher = poisoned model ĐÃ prune channel poison (detoxify).'''
    m = build_model(build_cfg(POISONED_WEIGHTS)); DetectionCheckpointer(m).load(POISONED_WEIGHTS)
    m.to(DEVICE)
    if prune_frac>0: m,_ = collect_and_prune(m, prune_frac, prune_box=prune_box)
    m.eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m
print('cls_logits + load_teacher_detox defined.')

cls_logits + load_teacher_detox defined.


## 10. Self-distillation trainer
- **Distill (giữ):** trên ảnh test SẠCH, student khớp logits teacher (MSE) → giữ hành vi clean, retention.
- **Suppress (quên):** dán poison patch lên ảnh, ép student logits tại vùng patch xuống thấp → forget poison.
- Backbone freeze, chỉ head học. λ_sup cân bằng 2 lực.

In [14]:
def paste_poison_get_region(img, rng):
    im = img.copy(); regions=[]
    for _ in range(rng.integers(1,3)):
        patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
        y0=int(rng.integers(0,IMG_H-ph)); x0=int(rng.integers(0,IMG_W-pw))
        im[y0:y0+ph, x0:x0+pw] = patch; regions.append((x0,y0,x0+pw,y0+ph))
    return im, regions

def _region_mask_like(logit, regions, H, W):
    _,_,Hi,Wi = logit.shape; m = torch.zeros(1,1,Hi,Wi, device=logit.device)
    sy,sx = Hi/H, Wi/W
    for (x1,y1,x2,y2) in regions:
        a,b=int(y1*sy),max(int(y2*sy)+1,int(y1*sy)+1); cc,d=int(x1*sx),max(int(x2*sx)+1,int(x1*sx)+1)
        m[:,:,a:b,cc:d]=1.0
    return m

def train_distill(prune_teacher=0.05, n_test=250, lr=1e-4, iters=400, lam_sup=1.0,
                  lam_fvar=1.0, sup_ratio=0.5, lam_cos=0.0, seed=SEED, tag='d0'):
    out=WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    teacher = load_teacher_detox(prune_frac=prune_teacher)   # RobustKD: teacher ĐÃ detox
    student = build_model(build_cfg(POISONED_WEIGHTS, output_dir=out))
    DetectionCheckpointer(student).load(POISONED_WEIGHTS); student.to(DEVICE)
    for n,p in student.named_parameters(): p.requires_grad = not n.startswith('backbone.')
    opt=torch.optim.Adam([p for p in student.parameters() if p.requires_grad], lr=lr)
    anchor={n:p.detach().clone() for n,p in student.named_parameters() if p.requires_grad}  # ULRL: neo gốc
    rng=np.random.default_rng(seed)
    pick=rng.choice(len(test_files), size=min(n_test,len(test_files)), replace=False)
    imgs=[load_image(test_files[int(k)]) for k in pick]
    student.train()
    for step in range(iters):
        img = imgs[int(rng.integers(len(imgs)))]
        do_sup = rng.random() < sup_ratio
        if do_sup:
            im2, regions = paste_poison_get_region(img, rng)
            xt = torch.as_tensor(im2.transpose(2,0,1), dtype=torch.float32, device=DEVICE)
            tf=[]; sf=[]
            with torch.no_grad(): t_log = cls_logits(teacher, xt, tf)
            s_log = cls_logits(student, xt, sf)
            loss=0.0
            for tl,sl in zip(t_log,s_log):
                mask=_region_mask_like(sl, regions, IMG_H, IMG_W)
                loss=loss+((sl-tl)**2*(1-mask)).mean()                # khớp teacher_detox ngoài patch
                loss=loss+lam_sup*(torch.sigmoid(sl)**2*mask).mean()   # suppress trong patch
        else:
            xt = torch.as_tensor(img.transpose(2,0,1), dtype=torch.float32, device=DEVICE)
            tf=[]; sf=[]
            with torch.no_grad(): t_log = cls_logits(teacher, xt, tf)
            s_log = cls_logits(student, xt, sf)
            loss=sum(((sl-tl)**2).mean() for tl,sl in zip(t_log,s_log))
        # RobustKD feature-variance: giảm variance feature student vs teacher_detox
        if lam_fvar>0 and tf and sf:
            fvar=sum((sfi-tfi).var() for sfi,tfi in zip(sf,tf))/len(sf)
            loss=loss+lam_fvar*fvar
        if lam_cos>0:
            cos=torch.tensor(0.0,device=DEVICE)
            for n,p in student.named_parameters():
                if p.requires_grad and n in anchor:
                    a=anchor[n].flatten(); b=p.flatten()
                    cos=cos+(1-torch.nn.functional.cosine_similarity(a,b,dim=0))
            loss=loss+lam_cos*cos
        opt.zero_grad(); loss.backward(); opt.step()
        if (step+1)%100==0: print(f'  distill it{step+1}: loss={float(loss):.4f}')
    final=str(out/'model_final.pth'); torch.save({'model':student.state_dict()}, final)
    del teacher, student; torch.cuda.empty_cache()
    return final
print('train_distill (RobustKD) defined.')

train_distill (RobustKD) defined.


## 11. Sweep distillation — chọn theo val_macadd

In [15]:
SWEEP=[
    dict(prune_teacher=0.05, n_test=400, lr=1e-4, iters=500, lam_sup=0.5, lam_fvar=3.0, sup_ratio=0.35, lam_cos=0.5, seed=42, tag='v0'),
    dict(prune_teacher=0.05, n_test=400, lr=1e-4, iters=600, lam_sup=1.0, lam_fvar=2.0, sup_ratio=0.4,  lam_cos=1.0, seed=42, tag='v1'),
    dict(prune_teacher=0.05, n_test=500, lr=1e-4, iters=600, lam_sup=0.5, lam_fvar=4.0, sup_ratio=0.3,  lam_cos=0.5, seed=7,  tag='v2'),
    dict(prune_teacher=0.03, n_test=400, lr=1e-4, iters=500, lam_sup=0.5, lam_fvar=3.0, sup_ratio=0.35, lam_cos=1.0, seed=13, tag='v3'),
]
sweep_results=[]
for spec in SWEEP:
    w=train_distill(**spec)
    pr=build_predictor(w, CAND_THRESH); stats=eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    fid=conf_fidelity(pr2:=build_predictor(w,CAND_THRESH), VAL, CLEAN_REF); del pr2; torch.cuda.empty_cache()
    sweep_results.append({**spec,'weights':w,**stats,**{'fid_'+k:v for k,v in fid.items()}})
    print(f"{spec['tag']}: firing={stats['poison_firing_rate']:.3f} retention={stats['streak_retention']:.3f} val_macadd={stats['val_macadd']:.4f}")
    print(f"    conf_gap={fid['streak_conf_gap']:.4f} miss={fid['streak_miss_rate']:.3f} poison_resid={fid['poison_residual_conf']:.4f}")
sweep_df=pd.DataFrame(sweep_results).sort_values('val_macadd')
print('\n', sweep_df[['tag','n_test','lam_sup','lam_fvar','sup_ratio','lam_cos','poison_firing_rate','streak_retention','val_macadd']].to_string(index=False))
print(f"\nPOISONED: firing={base_stats['poison_firing_rate']:.3f} retention={base_stats['streak_retention']:.3f} val_macadd={base_stats['val_macadd']:.4f}")
print(f"neo NB9-v2: val_macadd=1.040 retention=0.54")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


  distill it100: loss=0.0114
  distill it200: loss=0.0070
  distill it300: loss=0.0056
  distill it400: loss=0.0045
  distill it500: loss=0.0044


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


v0: firing=0.313 retention=0.667 val_macadd=0.9863
    conf_gap=0.4141 miss=0.333 poison_resid=0.2608


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

v2: firing=0.252 retention=0.616 val_macadd=0.9998
    conf_gap=0.4388 miss=0.384 poison_resid=0.2557


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


  distill it100: loss=0.0109
  distill it200: loss=0.0072
  distill it300: loss=0.0056
  distill it400: loss=0.0047
  distill it500: loss=0.0033


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


v3: firing=0.706 retention=0.887 val_macadd=1.0289
    conf_gap=0.2684 miss=0.107 poison_resid=0.3307

 tag  n_test  lam_sup  lam_fvar  sup_ratio  lam_cos  poison_firing_rate  streak_retention  val_macadd
 v1     400      1.0       2.0       0.40      1.0            0.257669          0.654088    0.979949
 v0     400      0.5       3.0       0.35      0.5            0.312883          0.666667    0.986338
 v2     500      0.5       4.0       0.30      0.5            0.251534          0.616352    0.999754
 v3     400      0.5       3.0       0.35      1.0            0.705521          0.886792    1.028934

POISONED: firing=0.969 retention=1.000 val_macadd=1.4549
neo NB9-v2: val_macadd=1.040 retention=0.54


## 12. Chọn cấu hình + model soup

In [16]:
cands=[r for r in sweep_results if r['streak_retention']>=0.6 and r['val_macadd']<LB_ANCHOR_MACADD]
pool=cands if cands else sweep_results
best=min(pool, key=lambda r:r['val_macadd'])
soup_src=[r for r in sweep_results if r['val_macadd']<LB_ANCHOR_MACADD] or [best]
if len(soup_src)>=2:
    sds=[torch.load(r['weights'],map_location='cpu')['model'] for r in soup_src]
    avg={k:sum(sd[k].float() for sd in sds)/len(sds) for k in sds[0]}
    sw=str(WORK/'soup.pth'); torch.save({'model':avg}, sw)
    pr=build_predictor(sw,CAND_THRESH); ss=eval_model_on_val(pr,VAL); del pr; torch.cuda.empty_cache()
    print('SOUP:', {k:round(v,4) for k,v in ss.items()})
    if ss['streak_retention']>=0.6 and ss['val_macadd']<best['val_macadd']: best={'tag':'soup','weights':sw,**ss}
FINAL_WEIGHTS=best['weights']; FINAL_STATS={k:best[k] for k in ('poison_firing_rate','streak_retention','val_macadd')}
print('CHỌN:', best['tag'], {k:round(float(v),4) for k,v in FINAL_STATS.items()})

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


SOUP: {'poison_firing_rate': 0.3497, 'streak_retention': 0.7233, 'val_macadd': 0.9683}
CHỌN: soup {'poison_firing_rate': 0.3497, 'streak_retention': 0.7233, 'val_macadd': 0.9683}


## 13. TTA + quét SUBMIT_THRESH (tinh chỉnh cuối)

In [17]:
final_pr=build_predictor(FINAL_WEIGHTS, CAND_THRESH)
tta_stats=eval_model_on_val(final_pr, VAL, tta=True)
USE_TTA = tta_stats['val_macadd'] <= FINAL_STATS['val_macadd']
print('no-TTA:', {k:round(float(v),4) for k,v in FINAL_STATS.items()}, '| TTA:', {k:round(v,4) for k,v in tta_stats.items()}, '-> TTA:', USE_TTA)
def eval_at_thresh(predictor, val, thr):
    pk={}
    for i,item in enumerate(val):
        b,s=predict_image(predictor, item['img']); keep=s>thr
        pk[str(i)]=(b[keep],s[keep]) if len(b) else (b,s)
    return macadd(CLEAN_REF, pk)
grid=[0.2,0.25,0.3,0.35,0.4,0.45,0.5]
tv=[(t,eval_at_thresh(final_pr,VAL,t)) for t in grid]
for t,v in tv: print(f'thr={t}: val_macadd={v:.4f}')
BEST_THRESH=min(tv,key=lambda x:x[1])[0]; print('-> BEST_THRESH=',BEST_THRESH)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


no-TTA: {'poison_firing_rate': 0.3497, 'streak_retention': 0.7233, 'val_macadd': 0.9683} | TTA: {'poison_firing_rate': 0.4356, 'streak_retention': 0.6855, 'val_macadd': 1.0718} -> TTA: False
thr=0.2: val_macadd=0.9683
thr=0.25: val_macadd=0.9550
thr=0.3: val_macadd=0.9615
thr=0.35: val_macadd=0.9635
thr=0.4: val_macadd=0.9823
thr=0.45: val_macadd=1.0302
thr=0.5: val_macadd=1.1063
-> BEST_THRESH= 0.25


## 14. Submission = output trực tiếp student distilled

In [18]:
sample=pd.read_csv(SAMPLE_SUB, dtype={'image_id':str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f'{float(sc):.6f}',f'{x1:.2f}',f'{y1:.2f}',f'{w:.2f}',f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred={}
for p in tqdm(test_files, desc='infer test'):
    img=load_image(p); b,s=predict_tta(final_pr,img) if USE_TTA else predict_image(final_pr,img)
    keep=s>BEST_THRESH; pred[p.stem]=(b[keep],s[keep])
sub=sample.copy(); sub['prediction_string']=sub['image_id'].map(lambda i: fmt(*pred.get(str(i),E)))
sub['prediction_string']=sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string']=' '
sub.to_csv(SUBMISSION_PATH, index=False)
total=sum(len(s) for _,s in pred.values()); npred=int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | dets={total} ({total/len(test_files):.3f}/img) @thr={BEST_THRESH} | imgs w/ preds: {npred}')

infer test: 100%|██████████| 2000/2000 [03:56<00:00,  8.46it/s]

saved /kaggle/working/submission.csv | dets=667 (0.334/img) @thr=0.25 | imgs w/ preds: 554


## 15. Tổng kết + quy tắc nộp (co-author-grade)

In [19]:
print('='*60)
print('POISONED   :', {k:round(v,4) for k,v in base_stats.items()})
print('DE-POISONED:', {k:round(float(v),4) for k,v in FINAL_STATS.items()})
final_fid=conf_fidelity(build_predictor(FINAL_WEIGHTS,CAND_THRESH), VAL, CLEAN_REF, tta=USE_TTA)
print('conf-fidelity :', {k:round(v,4) for k,v in final_fid.items()})
print(f"  streak_conf_gap {base_fid['streak_conf_gap']:.4f} -> {final_fid['streak_conf_gap']:.4f} | poison_resid {base_fid['poison_residual_conf']:.4f} -> {final_fid['poison_residual_conf']:.4f}")
print('neo NB4-s1 : val_macadd=%.3f (LB=258.7)' % 1.040)
firing_drop = base_stats['poison_firing_rate'] - FINAL_STATS['poison_firing_rate']
better = FINAL_STATS['val_macadd'] < 1.040
ret_ok = FINAL_STATS['streak_retention'] >= 0.65
real = firing_drop >= 0.3
ok = better and ret_ok and real
print(f'\nfiring giảm thật: {firing_drop:.3f} | retention ok: {ret_ok} | beat neo: {better}')
print('ĐẠT TIÊU CHÍ NỘP (co-author-grade):', ok)
if ok:
    print(' -> NÊN nộp: self-distillation de-poison thật, giữ retention qua khớp teacher, đúng định nghĩa metric.')
else:
    print(' -> CHƯA đạt. Tăng sup_ratio/lam_sup nếu firing còn cao; giảm nếu retention rớt; tăng iters.')

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


POISONED   : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
DE-POISONED: {'poison_firing_rate': 0.3497, 'streak_retention': 0.7233, 'val_macadd': 0.9683}
conf-fidelity : {'streak_conf_gap': 0.3853, 'streak_miss_rate': 0.2767, 'pred_conf_on_streak': 0.3919, 'ref_conf_on_streak': 0.756, 'poison_residual_conf': 0.2809, 'poison_fire_frac': 0.3497}
  streak_conf_gap 0.0194 -> 0.3853 | poison_resid 0.5937 -> 0.2809
neo NB4-s1 : val_macadd=1.040 (LB=258.7)

firing giảm thật: 0.620 | retention ok: True | beat neo: True
ĐẠT TIÊU CHÍ NỘP (co-author-grade): True
 -> NÊN nộp: self-distillation de-poison thật, giữ retention qua khớp teacher, đúng định nghĩa metric.
